# Lecture 04: Advanced Transformations, Data Cleaning & Vectorized Pandas UDFs
This notebook provides a professional guide to advanced data cleaning operations, date/time manipulation, regex string parsing, standard Python UDFs, and Arrow-optimized Vectorized Pandas UDFs.

### 1. Initialize SparkSession
We configure the session enabling Apache Arrow execution to accelerate data exchange between PySpark and Pandas.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Lecture04_Transforms") \
    .master("local[2]") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("WARN")
print("Spark Session initialized with Apache Arrow enabled!")

### 2. Data Cleaning: Dropping Null Values
Demonstrating how to drop rows containing any nulls, all nulls, or nulls in specific columns using `.na.drop()`.

In [ ]:
dirty_data = [
    (1, "Alice", None),
    (2, None, 85.5),
    (3, "Charlie", 92.0),
    (None, None, None)
]
df_dirty = spark.createDataFrame(dirty_data, ["id", "name", "score"])

# Drop rows where all elements are null
df_dirty.na.drop(how="all").show()

# Drop rows where 'name' or 'score' is null
df_dirty.na.drop(subset=["name", "score"]).show()

### 3. Data Cleaning: Filling Default Values
Filling missing values matching datatype maps using `.na.fill()` to avoid null pointer issues during calculation.

In [ ]:
df_filled = df_dirty.na.fill({
    "id": 0,
    "name": "Unknown_User",
    "score": 0.0
})
df_filled.show()

### 4. Data Cleaning: Value Replacements
Replacing placeholder or incorrect values using a translation dictionary with `.na.replace()`.

In [ ]:
df_replaced = df_filled.na.replace({"Unknown_User": "Guest_User"})
df_replaced.show()

### 5. String to Timestamp Parsing
Converting string dates with specific formats into Spark TimestampType columns using `to_timestamp()`.

In [ ]:
from pyspark.sql.functions import to_timestamp, col

date_df = spark.createDataFrame([("05-06-2026 18:00:00",)], ["date_str"])
parsed_df = date_df.withColumn(
    "event_timestamp",
    to_timestamp(col("date_str"), "dd-MM-yyyy HH:mm:ss")
)
parsed_df.show()
parsed_df.printSchema()

### 6. Formatting Timestamps back to Strings
Converting timestamps into custom string representations using `date_format()`.

In [ ]:
from pyspark.sql.functions import date_format, current_timestamp

spark.range(1).select(
    current_timestamp().alias("now"),
    date_format(current_timestamp(), "yyyy/MM/dd HH:mm").alias("formatted")
).show(truncate=False)

### 7. Date shifts Arithmetic
Using `date_add()` and `date_sub()` to shift calendar dates by offset days.

In [ ]:
from pyspark.sql.functions import date_add, date_sub, lit

spark.range(1).select(
    date_add(lit("2026-06-05"), 7).alias("next_week"),
    date_sub(lit("2026-06-05"), 7).alias("last_week")
).show()

### 8. Date Differences (datediff)
Calculating the exact number of days elapsed between two dates.

In [ ]:
from pyspark.sql.functions import datediff, lit

spark.range(1).select(
    datediff(lit("2026-06-15"), lit("2026-06-05")).alias("days_gap")
).show()

### 9. Epoch/Unix Timestamps conversions
Working with Epoch seconds (unix timestamps) to calculate numeric time intervals.

In [ ]:
from pyspark.sql.functions import unix_timestamp, from_unixtime

spark.range(1).select(
    unix_timestamp().alias("epoch_seconds"),
    from_unixtime(lit(1780000000)).alias("date_representation")
).show(truncate=False)

### 10. Timezone conversions
Using `from_utc_timestamp()` and `to_utc_timestamp()` to align event times to local regions (e.g. EST, Cairo).

In [ ]:
from pyspark.sql.functions import from_utc_timestamp, to_utc_timestamp

spark.range(1).select(
    current_timestamp().alias("utc_now"),
    from_utc_timestamp(current_timestamp(), "America/New_York").alias("est_now"),
    to_utc_timestamp(current_timestamp(), "Africa/Cairo").alias("cairo_now")
).show(truncate=False)

### 11. regexp_extract Column parsing
Extracting matched patterns from unstructured fields (e.g. parsing device error codes).

In [ ]:
from pyspark.sql.functions import regexp_extract

logs_df = spark.createDataFrame([
    ("device_id:D102 - status:active",),
    ("device_id:D105 - status:error",)
], ["raw_log"])

logs_df.select(
    regexp_extract(col("raw_log"), r"device_id:(\w+)", 1).alias("device"),
    regexp_extract(col("raw_log"), r"status:(\w+)", 1).alias("status")
).show()

### 12. String split
Tokenizing strings into list arrays.

In [ ]:
from pyspark.sql.functions import split

names_df = spark.createDataFrame([("John-Doe-Smith",)], ["full_name"])
names_df.select(split(col("full_name"), "-").alias("name_parts")).show()

### 13. concat_ws
Joining multiple string fields with a custom separator.

In [ ]:
from pyspark.sql.functions import concat_ws, lit

spark.range(1).select(
    concat_ws(" ", lit("PySpark"), lit("Advanced"), lit("Transforms")).alias("concatenated")
).show()

### 14. when/otherwise conditional logic
Applying multi-level logic decisions to assign values dynamically.

In [ ]:
from pyspark.sql.functions import when

score_df = spark.createDataFrame([(45,), (85,), (92,)], ["score"])
score_df.select(
    col("score"),
    when(col("score") >= 90, "A") \
    .when(col("score") >= 80, "B") \
    .otherwise("C").alias("grade")
).show()

### 15. Defining a Standard Python UDF
We write a custom Python function to classify scores and register it as a PySpark UDF. Standard UDFs execute row-by-row and suffer from JVM-to-Python serialization overhead.

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def score_classifier(score):
    if score is None: return "Pending"
    return "Pass" if score >= 50 else "Fail"

python_udf = udf(score_classifier, StringType())
print("Standard Python scalar UDF defined and initialized.")

### 16. Evaluating Python UDF
Executing our UDF on a DataFrame column.

In [ ]:
test_scores = spark.createDataFrame([(45,), (None,), (75,)], ["score"])
test_scores.withColumn("status", python_udf("score")).show()

### 17. Vectorized Pandas UDF (Arrow Integration)
Pandas UDFs are built on Apache Arrow to enable vectorized execution, which processes batches of rows using Pandas/NumPy in Python rather than iterating row-by-row.

In [ ]:
from pyspark.sql.functions import pandas_udf
import pandas as pd

@pandas_udf(StringType())
def pandas_score_classifier(scores: pd.Series) -> pd.Series:
    # Vectorized execution using pandas
    return scores.apply(lambda x: "Pass" if (pd.notnull(x) and x >= 50) else ("Pending" if pd.isnull(x) else "Fail"))

print("Vectorized Pandas UDF registered successfully!")

### 18. Evaluating Vectorized Pandas UDF
Running the Pandas UDF on our scores dataset.

In [ ]:
test_scores.withColumn("pandas_status", pandas_score_classifier("score")).show()

### 19. Generate Large Dataset for Benchmarking
Creating a dataset with 1,000,000 values to compare performance between UDF implementations.

In [ ]:
large_df = spark.range(1, 1000000).withColumn("val", col("id").cast("double"))
print("Generated large dataframe with partition size:", large_df.rdd.getNumPartitions())

### 20. Execution Time Benchmark
Comparing standard Python UDF execution time against Vectorized Pandas UDF execution time.

In [ ]:
import time

# Standard Python UDF run
start_time = time.time()
large_df.withColumn("res", python_udf("val")).write.mode("overwrite").parquet("data/bench_py_udf")
duration_py = time.time() - start_time

# Pandas Vectorized UDF run
start_time = time.time()
large_df.withColumn("res", pandas_score_classifier("val")).write.mode("overwrite").parquet("data/bench_pandas_udf")
duration_pandas = time.time() - start_time

print(f"Standard Python UDF Time: {duration_py:.4f} seconds")
print(f"Vectorized Pandas UDF Time: {duration_pandas:.4f} seconds")
print(f"Arrow Vectorization speedup: {duration_py / duration_pandas:.2f}x")

### 21. Clean up resources
Stopping Spark Session.

In [ ]:
spark.stop()